In [1]:
import sys
import os
import pickle
import numpy as np
import pandas as pd
import MDAnalysis as mda
from MDAnalysis.analysis.hydrogenbonds import WaterBridgeAnalysis

/home/irati/miniconda3/envs/thesis_md/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
datadir = '../../data/1bf5/wt/rep2/analysis'

In [3]:

with open(f'{datadir}/000.raw/universe', 'rb') as f:
    u = pickle.load(f)
u.trajectory[0]

< Timestep 0 with unit cell dimensions [162.50388 162.50389 162.5039  109.47124 109.47121 109.47121] >

In [100]:

# Define groups
dimer = u.select_atoms("segid A B") 
dna = u.select_atoms("segid C D")

print("Dimer atoms:", len(dimer))
print("DNA atoms:", len(dna))

# Convert AtomGroup indices to selection strings
dimer_indices = " ".join(map(str, dimer.indices))
dna_indices = " ".join(map(str, dna.indices))

# Create selection strings
dimer_sel = f"index {dimer_indices}"
dna_sel = f"index {dna_indices}"

wba = WaterBridgeAnalysis(
    universe=u,
    selection1='protein or resname PTR',  
    selection2='nucleic',  
    water_selection="resname WAT",  
    distance=5,  
    angle=120,     
)

# Run the analysis over the trajectory
wba.run(start=1800, stop=2001, verbose=True)

Dimer atoms: 18826
DNA atoms: 1457


100%|██████████| 201/201 [04:29<00:00,  1.34s/it]


In [101]:
# Access results
results = wba.results.timeseries

In [102]:
results

[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],


In [103]:
# Generate the table
table = wba.generate_table()

# Convert to Pandas DataFrame for easy CSV export
df = pd.DataFrame(table)

In [99]:
df.head()

,time,sele1_index,sele2_index,sele1_resnm,sele1_resid,sele1_atom,sele2_resnm,sele2_resid,sele2_atom,distance,angle


In [76]:
# Save to CSV
# TODO check that the saved numbers are all integers (atom ids) and if so switch to fmt="%d"
df.to_csv(f'{datadir}/002.hbonds/wb_test.csv')

In [104]:
# Create selection strings
dimer_sel = f"index {dimer_indices}"
dna_sel = f"index {dna_indices}"

wba = WaterBridgeAnalysis(
    universe=u,
    selection1='protein',  
    selection2='not protein and not resname WAT',  
    water_selection="resname WAT",  
    distance=5,  
    angle=120,     
)

# Run the analysis over the trajectory
wba.run(start=1800, stop=2001, verbose=True)

100%|██████████| 201/201 [07:54<00:00,  2.36s/it]


In [105]:
# Generate the table
table = wba.generate_table()

# Convert to Pandas DataFrame for easy CSV export
df = pd.DataFrame(table)

In [107]:
df.shape

(1958, 11)

In [109]:
df.head(15)

,time,sele1_index,sele2_index,sele1_resnm,sele1_resid,sele1_atom,sele2_resnm,sele2_resid,sele2_atom,distance,angle
0,180000.0,7707,18672,ARG,467,HH22,PTR,566,OH,3.606247,154.647514
1,180000.0,7764,18672,SER,471,HG,PTR,566,OH,3.720604,150.522753
2,180000.0,16809,9259,LYS,449,HZ3,PTR,566,OH,3.975487,166.743664
3,180000.0,17117,9259,ARG,467,HH12,PTR,566,OH,3.278506,154.313886
4,180000.0,17120,9259,ARG,467,HH22,PTR,566,OH,3.561041,147.980332
5,180000.0,17151,9259,SER,469,HG,PTR,566,OH,2.498264,122.595773
6,180000.0,18645,18678,THR,564,HG1,PTR,566,O,4.428345,125.858049
7,180000.0,9234,9243,THR,564,O,PTR,566,H,2.732053,130.401954
8,180000.0,18633,18656,GLY,563,O,PTR,566,H,4.815243,121.150571
9,180100.0,7707,18672,ARG,467,HH22,PTR,566,OH,3.635961,137.564272


In [12]:
class WaterBridgeAnalysis_OtherFF(WaterBridgeAnalysis):
      DEFAULT_DONORS = {"OtherFF": tuple(set(["O", "H2", "N", "NE", "NH1", "NH2", "ND2", "SG", "NE2", "ND1", 
                                              "NZ", "OG", "OG1", "OH"]))}
      DEFAULT_ACCEPTORS = {"OtherFF": tuple(set(["O", "H2", "OC1", "OC2", "OD1", "OD2", 
                                                 "SG", "OE1", "OE2", "ND1", "NE2", "SD", "OG", "OG1", "OH"]))}

In [13]:
wba = WaterBridgeAnalysis_OtherFF(
    universe=u,
    selection1='protein or resname PTR',  
    selection2='nucleic',  
    water_selection="resname WAT",  
    forcefield="OtherFF",
    distance=5,  
    angle=120,     
)

# Run the analysis over the trajectory
wba.run(start=1900, stop=2001, verbose=True)

100%|██████████| 101/101 [04:04<00:00,  2.42s/it]


In [14]:
# Generate the table
table = wba.generate_table()

# Convert to Pandas DataFrame for easy CSV export
df = pd.DataFrame(table)

In [18]:
df.sort_values('time').head(15)

,time,sele1_index,sele2_index,sele1_resnm,sele1_resid,sele1_atom,sele2_resnm,sele2_resid,sele2_atom,distance,angle
0,190000.0,4618,261789,LYS,275,HZ2,WAT,60952,H2,4.415187,136.105871
3,190000.0,4671,261789,LYS,278,HZ1,WAT,60952,H2,4.436847,152.682869
1,190000.0,261789,20051,WAT,60952,H2,DA,16,H2,4.948638,137.238206
4,190000.0,4671,261787,LYS,278,HZ1,WAT,60952,O,4.020203,153.981700
2,190000.0,4618,261787,LYS,275,HZ2,WAT,60952,O,4.131767,143.300877
7,190100.0,4671,144529,LYS,278,HZ1,WAT,31637,H2,4.411927,136.444772
5,190100.0,4671,144527,LYS,278,HZ1,WAT,31637,O,3.760054,128.485878
6,190100.0,144529,20083,WAT,31637,H2,DA,17,H2,3.274353,126.966363
8,190200.0,4617,413201,LYS,275,HZ1,WAT,98805,H2,3.830010,131.423347
9,190200.0,413200,20083,WAT,98805,H1,DA,17,H2,4.114073,139.059187


In [16]:
df.shape

(545, 11)

In [17]:
wba.results.timeseries

[[(np.int64(4618),
   np.int64(261789),
   ('LYS', np.int64(275), 'HZ2'),
   ('WAT', np.int64(60952), 'H2'),
   np.float64(4.415186850547841),
   np.float64(136.10587055466743)),
  (np.int64(261789),
   np.int64(20051),
   ('WAT', np.int64(60952), 'H2'),
   ('DA', np.int64(16), 'H2'),
   np.float64(4.948637863165775),
   np.float64(137.23820556672226)),
  (np.int64(4618),
   np.int64(261787),
   ('LYS', np.int64(275), 'HZ2'),
   ('WAT', np.int64(60952), 'O'),
   np.float64(4.131767327817819),
   np.float64(143.30087721154507)),
  (np.int64(4671),
   np.int64(261789),
   ('LYS', np.int64(278), 'HZ1'),
   ('WAT', np.int64(60952), 'H2'),
   np.float64(4.436846615079838),
   np.float64(152.6828690433002)),
  (np.int64(4671),
   np.int64(261787),
   ('LYS', np.int64(278), 'HZ1'),
   ('WAT', np.int64(60952), 'O'),
   np.float64(4.020203262430828),
   np.float64(153.98170029150123))],
 [(np.int64(4671),
   np.int64(144527),
   ('LYS', np.int64(278), 'HZ1'),
   ('WAT', np.int64(31637), 'O'),
 